# Direct BPS Optimization — Cross-Currency Pooled Training

Train one DirectBPSModel on **all 7 currencies combined** (~4000 instruments)
instead of per-currency (~564).

**Hypothesis:** Per-currency training had only ~564 instruments for 721K parameters,
leading to memorization. Pooling provides ~4000 samples, potentially enabling the
encoder to learn general execution patterns that transfer across markets.

**Key changes from per-currency notebook:**
- Load all 7 currencies, process each separately, concatenate tensors
- Per-instrument `volume_to_fill` (model outputs softmax weights × each instrument's volume)
- Evaluate per-currency at the end

In [ ]:
import os, sys, subprocess

if not os.path.isdir("/content/Ultramarin/utils"):
    subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                    "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                   cwd="/content")
os.chdir("/content/Ultramarin")
sys.path.insert(0, os.getcwd())

import gc
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from data.simulate_walk_the_book import simulate_walk_the_book
from utils.utils import (
    compute_ofi, ASK_PRICE_COLS, ASK_VOL_COLS,
    BID_PRICE_COLS, BID_VOL_COLS, df_to_tensor,
)
from utils.datastuff import TrainCfg, LOBProcessor
from utils.train import chrono_split
from models.DeepLOB import DeepLOBEncoder
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
dst = "/content/Ultramarin/data"
if not os.path.isfile(os.path.join(dst, "BTCUSDT/X_train.parquet")):
    from google.colab import drive
    drive.mount('/content/drive')
    src = "/content/drive/MyDrive/data"
    os.makedirs(dst, exist_ok=True)
    subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
    print("Data copied.")
else:
    print("Data already present.")

In [ ]:
CURRENCIES = ["BTCUSDT", "ETHUSDT", "LTCUSDT", "SOLUSDT", "ADAUSDT", "DOGEUSDT", "XRPUSDT"]

KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}

# Logit bias initialization: median K across currencies
INIT_K = 17
INPUT_WINDOW = 600

# Set to True to freeze encoder and train only the 60 logit_bias values
BIAS_ONLY = False

root = Path("data")
print(f"Currencies: {len(CURRENCIES)}")
print(f"Init K: {INIT_K}")
print(f"Mode: {'BIAS-ONLY (60 params)' if BIAS_ONLY else 'FULL MODEL (721K params)'}")

In [ ]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

OFI_COLS = ['ofi_1', 'ofi_2', 'ofi_3', 'ofi_4', 'ofi_5', 'ofi_agg']
FEATURE_COLS = LOB_COLS + ["mid_price"] + OFI_COLS
print(f"Features: {len(FEATURE_COLS)}")

In [ ]:
def extract_clean_book_tensors(y_df, processor):
    y_pl = pl.from_pandas(y_df)
    y_clean = processor._apply_cleaning(y_pl)
    y_tensor, y_id_map = df_to_tensor(y_clean, seq_len=60)

    exclude = ["anonymized_id", "time_in_hour"]
    col_names = [c for c in y_clean.columns if c not in exclude]
    col_map = {name: i for i, name in enumerate(col_names)}

    return {
        "ask_prices": y_tensor[:, :, [col_map[c] for c in ASK_PRICE_COLS]],
        "ask_vols":   y_tensor[:, :, [col_map[c] for c in ASK_VOL_COLS]],
        "bid_prices": y_tensor[:, :, [col_map[c] for c in BID_PRICE_COLS]],
        "bid_vols":   y_tensor[:, :, [col_map[c] for c in BID_VOL_COLS]],
        "close_prices": y_tensor[-1, :, col_map["close"]],
        "id_map": y_id_map,
    }


def align_x_y(X_tensor, x_id_map, book_data):
    x_ids = {int(x_id_map[i, 1].item()): i for i in range(x_id_map.shape[0])}
    y_ids = {int(book_data['id_map'][i, 1].item()): i
             for i in range(book_data['id_map'].shape[0])}

    common = sorted(set(x_ids) & set(y_ids))
    xi = [x_ids[u] for u in common]
    yi = [y_ids[u] for u in common]

    return (
        X_tensor[:, xi, :],
        {k: v[:, yi, :] if v.dim() == 3 else v[yi]
         for k, v in book_data.items() if k != 'id_map'},
        common,
    )

print("Helper functions defined.")

In [ ]:
all_X_train, all_X_val = [], []
all_train_book, all_val_book = [], []
train_vtf_list, val_vtf_list = [], []
train_currency_list, val_currency_list = [], []

for asset in CURRENCIES:
    print(f"\n{'='*40}")
    print(f"Processing {asset}...")

    X_raw = pd.read_parquet(root / asset / "X_train.parquet").sort_values(
        ["anonymized_id", "time_in_hour"])
    Y_raw = pd.read_parquet(root / asset / "y_train.parquet").sort_values(
        ["anonymized_id", "time_in_hour"])

    x_train_df, x_val_df, y_train_df, y_val_df = chrono_split(X_raw, Y_raw, val_ratio=0.2)
    del X_raw, Y_raw

    for df in [x_train_df, x_val_df, y_train_df, y_val_df]:
        df["mid_price"] = (df["ask_price_1"] + df["bid_price_1"]) / 2.0

    x_train_df = compute_ofi(x_train_df)
    x_val_df = compute_ofi(x_val_df)
    y_train_df = compute_ofi(y_train_df)
    y_val_df = compute_ofi(y_val_df)

    processor = LOBProcessor(TrainCfg(), device=device)
    train_out = processor.process(pl.from_pandas(x_train_df), pl.from_pandas(y_train_df))
    val_out = processor.process(pl.from_pandas(x_val_df), pl.from_pandas(y_val_df))

    feat_idx = [processor.feature_map[col] for col in FEATURE_COLS]
    X_tr = train_out["X"][:, :, feat_idx]
    X_va = val_out["X"][:, :, feat_idx]

    tr_book = extract_clean_book_tensors(y_train_df, processor)
    va_book = extract_clean_book_tensors(y_val_df, processor)

    X_tr, tr_book, _ = align_x_y(X_tr, train_out["X_id_map"], tr_book)
    X_va, va_book, _ = align_x_y(X_va, val_out["X_id_map"], va_book)

    n_tr, n_va = X_tr.shape[1], X_va.shape[1]

    all_X_train.append(X_tr)
    all_X_val.append(X_va)
    all_train_book.append(tr_book)
    all_val_book.append(va_book)
    train_vtf_list.extend([KNOWN_VOLUMES[asset]] * n_tr)
    val_vtf_list.extend([KNOWN_VOLUMES[asset]] * n_va)
    train_currency_list.extend([asset] * n_tr)
    val_currency_list.extend([asset] * n_va)

    print(f"  Train: {n_tr}, Val: {n_va}")

    del x_train_df, x_val_df, y_train_df, y_val_df, train_out, val_out, processor
    gc.collect()

# Concatenate along instrument dimension
X_train = torch.cat(all_X_train, dim=1)
X_val = torch.cat(all_X_val, dim=1)

book_keys = ["ask_prices", "ask_vols", "bid_prices", "bid_vols"]
train_book = {k: torch.cat([b[k] for b in all_train_book], dim=1) for k in book_keys}
train_book["close_prices"] = torch.cat([b["close_prices"] for b in all_train_book], dim=0)
val_book = {k: torch.cat([b[k] for b in all_val_book], dim=1) for k in book_keys}
val_book["close_prices"] = torch.cat([b["close_prices"] for b in all_val_book], dim=0)

train_vtf = torch.tensor(train_vtf_list, dtype=torch.float32)
val_vtf = torch.tensor(val_vtf_list, dtype=torch.float32)

del all_X_train, all_X_val, all_train_book, all_val_book
gc.collect()

print(f"\n{'='*40}")
print(f"POOLED DATA:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
for asset in CURRENCIES:
    n_tr = train_currency_list.count(asset)
    n_va = val_currency_list.count(asset)
    print(f"  {asset}: {n_tr} train, {n_va} val")

In [ ]:
class DirectBPSDataset(Dataset):
    def __init__(self, X_tensor, book_data, volume_to_fill, input_window=600):
        self.X = X_tensor
        self.ask_prices = book_data["ask_prices"]
        self.ask_vols = book_data["ask_vols"]
        self.bid_prices = book_data["bid_prices"]
        self.bid_vols = book_data["bid_vols"]
        self.close_prices = book_data["close_prices"]
        self.volume_to_fill = volume_to_fill
        self.input_window = input_window

    def __len__(self):
        return self.X.shape[1]

    def __getitem__(self, idx):
        return (
            self.X[-self.input_window:, idx, :],
            self.ask_prices[:, idx, :],
            self.ask_vols[:, idx, :],
            self.bid_prices[:, idx, :],
            self.bid_vols[:, idx, :],
            self.close_prices[idx],
            self.volume_to_fill[idx],
        )


train_dataset = DirectBPSDataset(X_train, train_book, train_vtf, INPUT_WINDOW)
val_dataset = DirectBPSDataset(X_val, val_book, val_vtf, INPUT_WINDOW)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset)} instruments")
print(f"Val:   {len(val_dataset)} instruments")

In [ ]:
def diff_walk_the_book(positions, ask_prices, ask_vols):
    B = positions.shape[0]
    total_cost = torch.zeros(B, device=positions.device)
    total_filled = torch.zeros(B, device=positions.device)
    carry = torch.zeros(B, device=positions.device)

    for t in range(60):
        remaining = positions[:, t] + carry
        carry = torch.zeros(B, device=positions.device)

        for level in range(5):
            price = torch.nan_to_num(ask_prices[:, t, level], nan=0.0)
            avail = torch.nan_to_num(ask_vols[:, t, level], nan=0.0)
            take = torch.clamp(torch.minimum(remaining, avail), min=0.0)
            total_cost = total_cost + take * price
            total_filled = total_filled + take
            remaining = torch.clamp(remaining - take, min=0.0)

        carry = remaining

    vwap = total_cost / (total_filled + 1e-10)
    return vwap, total_filled


def bps_loss(positions, ask_prices, ask_vols, close_prices, volume_to_fill):
    """volume_to_fill is now [B] — per instrument."""
    vwap, total_filled = diff_walk_the_book(positions, ask_prices, ask_vols)
    bps_sq = ((vwap - close_prices) / (close_prices + 1e-10)) ** 2 * (10000 ** 2)
    fill_ratio = total_filled / (volume_to_fill + 1e-10)
    fill_penalty = torch.clamp(1.0 / (fill_ratio + 1e-10), max=100.0)
    loss = (bps_sq * fill_penalty).mean()
    with torch.no_grad():
        bps_actual = torch.abs(vwap - close_prices) / (close_prices + 1e-10) * 10000 * fill_penalty
    return loss, bps_actual

print("Differentiable walk-the-book defined.")

In [ ]:
class DirectBPSModel(nn.Module):
    def __init__(self, hidden=128, dropout=0.1, num_extra_features=0, twap_k=17):
        super().__init__()
        self.num_extra_features = num_extra_features
        enc_dim = 2 * hidden

        self.spatial = DeepLOBEncoder(in_ch=1, dropout=dropout)
        self.encoder = nn.LSTM(
            input_size=192 + num_extra_features, hidden_size=hidden,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.enc_dropout = nn.Dropout(dropout)

        self.queries = nn.Parameter(torch.randn(60, enc_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=enc_dim, num_heads=4, dropout=dropout, batch_first=True,
        )
        self.norm = nn.LayerNorm(enc_dim)

        self.head = nn.Sequential(
            nn.Linear(enc_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, 1),
        )

        self.logit_bias = nn.Parameter(torch.zeros(60))
        with torch.no_grad():
            self.logit_bias[:60 - twap_k] = -5.0

    def forward(self, x):
        B = x.shape[0]
        lob = x[:, :, :20]
        h = self.spatial(lob)

        if self.num_extra_features > 0:
            T_prime = h.shape[1]
            h = torch.cat([h, x[:, -T_prime:, 20:]], dim=-1)

        enc_out, _ = self.encoder(h)
        enc_out = self.enc_dropout(enc_out)

        q = self.queries.unsqueeze(0).expand(B, -1, -1)
        decoded, _ = self.cross_attn(q, enc_out, enc_out)
        decoded = self.norm(decoded + q)

        logits = self.head(decoded).squeeze(-1) + self.logit_bias
        return torch.softmax(logits, dim=1)


num_extra = len(FEATURE_COLS) - 20
model = DirectBPSModel(
    hidden=128, dropout=0.1, num_extra_features=num_extra, twap_k=INIT_K
).to(device)

if BIAS_ONLY:
    for name, param in model.named_parameters():
        if name != "logit_bias":
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,}")
print(f"Trainable: {trainable:,}")

In [ ]:
import copy
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 100
PATIENCE = 15

if BIAS_ONLY:
    LR, MIN_LR, WEIGHT_DECAY = 1e-2, 1e-4, 0.0
    params = [model.logit_bias]
else:
    LR, MIN_LR, WEIGHT_DECAY = 5e-4, 1e-6, 1e-5
    params = model.parameters()

optimizer = torch.optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)

best_val_bps = float("inf")
best_state = None
no_improve = 0

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_bps_all, train_loss_sum, train_n = [], 0.0, 0

    for x_b, ap_b, av_b, bp_b, bv_b, close_b, vtf_b in train_loader:
        x_b = x_b.to(device)
        ap_b = ap_b.to(device)
        av_b = av_b.to(device)
        close_b = close_b.to(device)
        vtf_b = vtf_b.to(device)

        optimizer.zero_grad()
        weights = model(x_b)
        positions = weights * vtf_b.unsqueeze(1)
        loss, bps_vals = bps_loss(positions, ap_b, av_b, close_b, vtf_b)
        loss.backward()
        if not BIAS_ONLY:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item() * x_b.size(0)
        train_bps_all.append(bps_vals.detach().cpu())
        train_n += x_b.size(0)

    train_bps = torch.cat(train_bps_all).mean().item()

    # ── Val ──
    model.eval()
    val_bps_all, val_n = [], 0

    with torch.no_grad():
        for x_b, ap_b, av_b, bp_b, bv_b, close_b, vtf_b in val_loader:
            x_b = x_b.to(device)
            ap_b = ap_b.to(device)
            av_b = av_b.to(device)
            close_b = close_b.to(device)
            vtf_b = vtf_b.to(device)

            weights = model(x_b)
            positions = weights * vtf_b.unsqueeze(1)
            _, bps_vals = bps_loss(positions, ap_b, av_b, close_b, vtf_b)
            val_bps_all.append(bps_vals.cpu())
            val_n += x_b.size(0)

    val_bps = torch.cat(val_bps_all).mean().item()
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch+1:02d} | Train BPS: {train_bps:.4f} | Val BPS: {val_bps:.4f} | LR: {lr:.2e}")

    if val_bps < best_val_bps:
        best_val_bps = val_bps
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nRestored best model (val BPS: {best_val_bps:.4f})")

## Evaluation — Per-Currency BPS

Compare pooled model vs per-currency TWAP-K using the numpy walk-the-book.

In [ ]:
model.eval()

val_currency_arr = np.array(val_currency_list)
val_vtf_arr = np.array(val_vtf_list)

model_bps_all = []
twap_bps_all = []
idx = 0

with torch.no_grad():
    for x_b, ap_b, av_b, bp_b, bv_b, close_b, vtf_b in val_loader:
        x_b = x_b.to(device)
        vtf_b_dev = vtf_b.to(device)
        weights = model(x_b)
        positions = (weights * vtf_b_dev.unsqueeze(1)).cpu().numpy()

        ap_np = ap_b.numpy()
        av_np = av_b.numpy()
        bp_np = bp_b.numpy()
        bv_np = bv_b.numpy()
        close_np = close_b.numpy()
        vtf_np = vtf_b.numpy()

        for i in range(positions.shape[0]):
            asset = val_currency_arr[idx]
            vtf = vtf_np[i]
            k = OPTIMAL_K[asset]

            # Model BPS
            vol, avg_price = simulate_walk_the_book(
                positions[i], ap_np[i], av_np[i], bp_np[i], bv_np[i]
            )
            if vol > 0 and not np.isnan(avg_price):
                ie = np.abs(avg_price - close_np[i]) / close_np[i] * 10000
                vp = min(100.0, vtf / vol)
                model_bps_all.append(ie * vp)
            else:
                model_bps_all.append(np.nan)

            # TWAP-K BPS
            twap_pos = np.zeros(60)
            twap_pos[-k:] = vtf / k
            vol_t, avg_t = simulate_walk_the_book(
                twap_pos, ap_np[i], av_np[i], bp_np[i], bv_np[i]
            )
            if vol_t > 0 and not np.isnan(avg_t):
                ie_t = np.abs(avg_t - close_np[i]) / close_np[i] * 10000
                vp_t = min(100.0, vtf / vol_t)
                twap_bps_all.append(ie_t * vp_t)
            else:
                twap_bps_all.append(np.nan)

            idx += 1

model_bps_arr = np.array(model_bps_all)
twap_bps_arr = np.array(twap_bps_all)
print(f"Evaluated {idx} instruments.")

In [ ]:
print(f"{'='*72}")
print(f"PER-CURRENCY RESULTS — {'BIAS-ONLY' if BIAS_ONLY else 'FULL MODEL'} (Pooled Training)")
print(f"{'='*72}")
print(f"{'Currency':<12} {'N':>5} {'Model BPS':>11} {'TWAP BPS':>11} {'vs TWAP':>10} {'Win%':>7}")
print(f"{'-'*72}")

for asset in CURRENCIES:
    mask = val_currency_arr == asset
    if mask.sum() == 0:
        continue
    m = np.nanmean(model_bps_arr[mask])
    t = np.nanmean(twap_bps_arr[mask])
    diff = t - m
    valid = ~np.isnan(model_bps_arr[mask]) & ~np.isnan(twap_bps_arr[mask])
    wins = (model_bps_arr[mask][valid] < twap_bps_arr[mask][valid]).sum()
    win_pct = wins / valid.sum() * 100 if valid.sum() > 0 else 0
    print(f"{asset:<12} {mask.sum():>5} {m:>11.4f} {t:>11.4f} {diff:>+10.4f} {win_pct:>6.1f}%")

print(f"{'-'*72}")
print(f"{'OVERALL':<12} {len(model_bps_arr):>5} {np.nanmean(model_bps_arr):>11.4f} "
      f"{np.nanmean(twap_bps_arr):>11.4f} "
      f"{np.nanmean(twap_bps_arr) - np.nanmean(model_bps_arr):>+10.4f}")
print(f"{'='*72}")

In [ ]:
# Per-currency results from direct_bps_results.md (vanilla, no init fix)
per_currency_bps = {
    "BTCUSDT": 1.2007,
    "ETHUSDT": 2.6233,
    "LTCUSDT": 5.5142,
    "SOLUSDT": 5.3452,
}

print(f"{'='*82}")
print(f"POOLED vs PER-CURRENCY vs TWAP")
print(f"{'='*82}")
print(f"{'Currency':<12} {'Pooled':>10} {'Per-Curr':>10} {'TWAP-K':>10} {'Pooled vs TWAP':>16} {'Per-Curr vs TWAP':>18}")
print(f"{'-'*82}")

for asset in CURRENCIES:
    mask = val_currency_arr == asset
    if mask.sum() == 0:
        continue
    pooled = np.nanmean(model_bps_arr[mask])
    twap = np.nanmean(twap_bps_arr[mask])
    pc = per_currency_bps.get(asset)
    pc_str = f"{pc:.4f}" if pc else "—"
    pc_diff = f"{twap - pc:+.4f}" if pc else "—"
    print(f"{asset:<12} {pooled:>10.4f} {pc_str:>10} {twap:>10.4f} {twap - pooled:>+16.4f} {pc_diff:>18}")

print(f"{'='*82}")
print()
print("Positive = model beats TWAP. Negative = TWAP wins.")
print("Per-Curr column: vanilla direct BPS trained on single currency (from direct_bps_results.md).")
print("ADA/DOGE/XRP were not tested per-currency.")